Começando a explorar a tabela orders e suas propriedades utlizando da biblioteca `csv` (nativa do **Python**).

Quero primeiro ler o arquivo csv e entender a estrutura dos dados, como colunas, tipos de dados e possíveis valores nulos. Em seguida, vou converter para um arquivo **.sqlite** para armazenar esses dados e realizar consultas **SQL** para analisar as informações contidas no arquivo utilizando `sqlite3` do **Python** e utilizando o **DatagGrip/DBeaver** de forma complementar.

In [1]:
"""Importando biblioteca nativa do python para manipulação de arquivos
csv e do sqlite3 para manipulação de banco de dados."""

import csv

# 1. definindo o caminho do arquivo de origem orders.csv
csv_path = r"..\..\lh_nautical_csv\orders.csv"


# 2. criando uma função para verificar o tipo de cada valor por coluna
def inferir_tipo(valor_entrada):
    """Função para inferir o tipo de dado de um valor de uma coluna."""
    if valor_entrada is None or valor_entrada == "":
        return "vazio"  # se o valor for nulo ou vazio, retorna "vazio"
    try:
        int(valor_entrada)
        return "inteiro"  # se o valor for um inteiro, retorna "inteiro"
    except ValueError:  # se o valor não for um inteiro, ignora e continua
        pass
    try:
        float(valor_entrada)
        return "decimal"  # se o valor for um decimal, retorna "decimal"
    except ValueError:  # se o valor não for um decimal, ignora e continua
        pass
    if valor_entrada.lower() in ("true", "false"):
        return "booleano"  # se o valor for um booleano, retorna "booleano"
    return "texto"  # se o valor não for nenhum dos tipos acima, retorna "texto"


# 3. criando uma função para formatar o valor de cada coluna
# para deixar num visual de tabela no print
def formatar_valor(valor_entrada, limite=30):
    """Diminui o texto e adiciona reticências caso ultrapasse o limite de 30 caracteres."""
    if valor_entrada is None or valor_entrada == "":
        return "<vazio>"  # se o valor for nulo ou vazio, retorna "<vazio>"
    texto = str(valor_entrada)
    if len(texto) > limite:
        return (
            texto[: limite - 3] + "..."
        )  # se o valor ultrapassar o limite, retorna os primeiros 27 caracteres + "..."
    return texto


# 4. recebendo o nome das colunas do arquivo csv e o primeiro valor
# não vazio de cada coluna por meio das funções inferir_tipo e formatar_valor
with open(csv_path, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)  # lendo o arquivo csv como um dicionário
    colunas = reader.fieldnames or []  # obtendo o nome das colunas do arquivo csv

    if not colunas:  # Printa se o arquivo csv está vazio ou não possui cabeçalho
        print("O arquivo CSV está vazio ou não possui cabeçalho.")
    else:
        primeiros_valores: dict[str, str | None] = {
            coluna: None for coluna in colunas
        }  # inicializando um dicionário para armazenar o primeiro valor não vazio de cada coluna

        for linha in reader:
            for coluna in colunas:
                if primeiros_valores[coluna] is None:
                    valor_linha = linha[coluna]
                    if valor_linha not in (None, ""):
                        primeiros_valores[coluna] = valor_linha
                        # se o valor da coluna for diferente de None ou
                        # vazio, armazena o valor no dicionário primeiros_valores
        # imprimindo o cabeçalho da tabela e o resumo das colunas
        # valores de exemplo e tipos de dados
        print("\nResumo da tabela orders\n")
        print("Quantidade de colunas:", len(colunas))
        print(f"{'Coluna':<25} | {'Valor de exemplo':<30} | Tipo")
        print("-" * 72)

        for coluna in colunas:
            valor_exemplo = primeiros_valores[coluna]
            print(
                f"{coluna:<25} | {formatar_valor(valor_exemplo):<30} | {inferir_tipo(valor_exemplo)}"
            )  # imprimindo o nome da coluna, o valor de exemplo e o tipo de dado


Resumo da tabela orders

Quantidade de colunas: 13
Coluna                    | Valor de exemplo               | Tipo
------------------------------------------------------------------------
id                        | 1                              | inteiro
order_number              | SO-000001                      | texto
channel                   | ecommerce                      | texto
customer_id               | 1136                           | inteiro
salesperson_id            | 9                              | inteiro
location_id               | 1                              | inteiro
status                    | paid                           | texto
subtotal                  | 323.34                         | decimal
discount_amount           | 35.57                          | decimal
total                     | 287.77                         | decimal
placed_at                 | 2022-09-06 05:37:37            | texto
created_at                | 2022-09-06 05:37:37           

Após primeira análise do arquivo **orders.csv**, percebi que ela possui 13 colunas, sendo elas:
- **id**: de valor do tipo inteiro e chave primária
- **order_number**: de valor do tipo texto
- **channel**: de valor do tipo texto
- **customer_id**: de valor do tipo inteiro
- **salesperson_id**: de valor do tipo real (real pois contém valores nulos no e-commerce)
- **location_id**: de valor do tipo inteiro
- **status**: de valor do tipo texto
- **subtotal**: de valor do tipo Real (real pois contém valores decimais)
- **discount_amount**: de valor do tipo Real (real pois contém valores decimais)
- **total**: de valor do tipo Real (real pois contém valores decimais)
- **placed_at**: de valor do tipo texto
- **created_at**: de valor do tipo texto
- **updated_at**: de valor do tipo texto

Com essas informações, posso prosseguir para a próxima etapa que é a conversão do arquivo `orders.csv` para um banco de dados `sqlite`.

In [2]:
import sqlite3

# 1. definindo o caminho do arquivo de destino orders.sqlite
db_path = "orders.sqlite"
csv_path = r"..\..\lh_nautical_csv\orders.csv"

# 2. Conecta (ou cria) o arquivo SQLite no disco
conn = sqlite3.connect(db_path)
cursor = conn.cursor()  # Cursor para executar comandos SQL
cursor.execute("DROP TABLE IF EXISTS orders;")  # Remove a tabela se ela já existir
# 3. Cria a tabela 'orders' no SQLite
cursor.execute("""
    CREATE TABLE IF NOT EXISTS orders (
        id INTEGER PRIMARY KEY,
        order_number TEXT,
        channel TEXT,
        customer_id INTEGER,
        salesperson_id REAL,  -- REAL pois contém nulos no e-commerce
        location_id INTEGER,
        status TEXT,
        subtotal REAL,
        discount_amount REAL,
        total REAL,
        placed_at TEXT,
        created_at TEXT,
        updated_at TEXT
    )
""")  # cria a tabela orders com as colunas e tipos de dados correspondentes ao CSV
# 4. Lê o CSV com o módulo nativo 'csv' e insere no banco
with open(csv_path, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    # Prepara os dados de strings vazias para None/NULL
    rows_to_insert = []
    for row in reader:
        rows_to_insert.append(
            (
                int(row["id"]),
                row["order_number"],
                row["channel"],
                int(row["customer_id"]),
                float(row["salesperson_id"]) if row["salesperson_id"] else None,
                int(row["location_id"]),
                row["status"],
                float(row["subtotal"]),
                float(row["discount_amount"]),
                float(row["total"]),
                row["placed_at"],
                row["created_at"],
                row["updated_at"],
            )  # Prepara os dados para inserção
        )
    # Insere todas as linhas de uma vez (muito mais rápido)
    cursor.executemany(
        """
        INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
        rows_to_insert,
    )
# 5. IMPORTANTE: Salva as alterações no arquivo físico e fecha
conn.commit()
conn.close()  # Salva as alterações no arquivo físico e fecha a conexão com o banco de dados
print(f"Sucesso! Banco SQLite salvo em: {db_path}")

Sucesso! Banco SQLite salvo em: orders.sqlite


Pronto! Com o arquivo sqlite criado, posso finalimente realizar consultas SQL para analisar os dados contidos na tabela orders. Vou realizar as consultas solicitadas na parte 1 e 2 das Tarefas do EDA e exportar as consultas para um script sql.

In [3]:
import sqlite3

# 1. Conecta para consultar
db_path = "orders.sqlite"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
# 2. Define a query SQL para obter informações da tabela 'orders' e depois salva em um arquivo .sql
QUERY_Q1 = """SELECT
    COUNT(*) AS total_linhas,
    MIN(created_at) AS data_minima,
    MAX(created_at) AS data_maxima,
    MIN(total) AS valor_minimo,
    MAX(total) AS valor_maximo,
    ROUND(AVG(total), 2) AS valor_medio
FROM orders;
"""
with open("consulta_q1_1.sql", mode="w", encoding="utf-8") as f:
    f.write(QUERY_Q1)
print("Sucesso! Arquivo 'consulta_q1_1.sql' exportado.\n")
# 3. Executa a query
cursor.execute(QUERY_Q1)
# 4. Extrai os nomes das colunas e os valores do resultado
nomes_colunas = [descricao[0] for descricao in cursor.description]
linha_resultado = cursor.fetchone()
# 5. Imprime de forma organizada no notebook
print("Consulta da Tabela 'orders' seguindo as Partes 1 e 2 das Tarefas do EDA")
print("-" * 72)
for coluna, valor in zip(nomes_colunas, linha_resultado):
    print(f"{coluna:<30} : {valor}")
# 6. Fecha a conexão
conn.close()

Sucesso! Arquivo 'consulta_q1_1.sql' exportado.

Consulta da Tabela 'orders' seguindo as Partes 1 e 2 das Tarefas do EDA
------------------------------------------------------------------------
total_linhas                   : 48998
data_minima                    : 2020-01-01 01:19:28
data_maxima                    : 2026-12-31 23:43:09
valor_minimo                   : 32.62
valor_maximo                   : 127262.02
valor_medio                    : 28704.99


Após realizar o EDA inicial, e já com o arquivo da consulta sql salvo, vejo que a tabela orders possui 48998 linhas, e que o valor mínimo total de pedido é 32,62 e o máximo de 127.262,02, o que acaba sendo esperado no mercado da LH Nautical, por vender no varejo nautico, que possui desde produtos como acessórios até motores e embarcações, o que acaba gerando uma grande variação de valores de pedidos.

Verificando no DBeaver também notei que a tabela possui pedidos de ID 50000, mas somente possui 48.998 linhas reais, 1002 lacunas de IDS, o que pode ocorrer em sistemas relacionais de ERP e e-commerce (como PostgreSQL), onde o id é gerado por uma sequência automática (Auto-Increment / Serial). 

As Consultas realizadas para verificar os IDs faltantes foram as seguintes:

```sql
SELECT -- Verificando a quantidade de IDS faltantes
    MIN(id) AS id_minimo,
    MAX(id) AS id_maximo,
    COUNT(id) AS total_registros,
    (MAX(id) - MIN(id) + 1 - COUNT(id)) AS total_ids_faltantes
FROM orders;

SELECT -- Listando os 10 primeiros IDS faltantes
    id + 1 AS id_faltante_inicio,
    proximo_id - 1 AS id_faltante_fim
FROM (
    SELECT 
        id, 
        LEAD(id) OVER (ORDER BY id) AS proximo_id
    FROM orders
)
WHERE proximo_id > id + 1
LIMIT 10;
```

Possíveis causas disso podem ser queda de conexão, falha no sistema ou até pagamento recusado. O contador do banco não volta atrás e acontece de pular números. Também podem ter sido utilizados para pedidos testes ou corrompidos e deletados posteriormente.

Quero agora analisar o pedido do Sr. Almir, e verificar se a tabela é confiável para análises futuras. Ver possíveis outliers no total, valores nulos ou inconsistentes e se precisa de alguma filtragem.

In [4]:
import sqlite3

# 1. Conecta para consultar
db_path = "orders.sqlite"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
# 2. Define a query SQL para obter informações da tabela 'orders'


def executar_diagnostico():
    """Função para executar o diagnóstico de confiabilidade dos dados na tabela orders"""
    print(
        "Questão 1 - Tarefa 3: Diagnóstico de confiabilidade dos dados na tabela 'orders'"
    )
    print("-" * 72)
    # 1. Canculando o Total de linhas para percentuais
    cursor.execute("SELECT COUNT(*) FROM orders;")
    total_linhas = cursor.fetchone()[0]

    # 2. Verificação de IDs duplicados
    cursor.execute(
        "SELECT COUNT(*) FROM (SELECT id FROM orders GROUP BY id HAVING COUNT(*) > 1);"
    )
    ids_duplicados = cursor.fetchone()[0]
    print("\n1. Verificação de Chave Primária ('id'):")
    print(f"   - IDs Duplicados: {ids_duplicados} (Garante unicidade dos registros)")

    # 3. Distribuição por Status de pedidos
    print("\n2. Distribuição de Pedidos por Status:")
    cursor.execute("""
        SELECT status, COUNT(*) AS total, ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS pct
        FROM orders
        GROUP BY status
        ORDER BY total DESC;
    """)
    for status, count, pct in cursor.fetchall():
        print(f"   - {status}: {count} ({pct:.2f}%)")

    # 4. Distribuição por Canal dos pedidos
    print("\n3. Distribuição por Canal ('channel'):")
    cursor.execute("""
        SELECT channel, COUNT(*) AS total, ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS pct
        FROM orders
        GROUP BY channel
        ORDER BY total DESC;
    """)
    for channel, count, pct in cursor.fetchall():
        print(f"   - {channel}: {count} ({pct:.2f}%)")

    # 4.1. Análise de integridade dos dados por Canal por não ter salesperson_id (vendedor) agregado
    print(
        "\n3.1. Análise de integridade dos dados por Canal por não ter salesperson_id agregado:"
    )
    cursor.execute("""
    SELECT 
        channel AS canal,
        SUM(CASE WHEN salesperson_id IS NULL THEN 1 ELSE 0 END) AS nulos_vendedor,
        ROUND(100.0 * SUM(CASE WHEN salesperson_id IS NULL THEN 1 ELSE 0 END) / (SELECT COUNT(*) FROM orders), 2) AS pct_do_total_da_base
    FROM orders
    GROUP BY channel;
    """)
    resultados = cursor.fetchall()
    for canal, nulos, pct in resultados:
        print(
            f"   - Canal: {canal:<10} | Sem Id de Vendedor: {nulos:<6} | % do"
            f" Total Geral de Pedidos: {pct}%"
        )

    # 5. Integridade Matemática do Faturamento dos pedidos
    cursor.execute("""
        SELECT MAX(ABS((subtotal - discount_amount) - total)) FROM orders;
    """)
    diferenca_maxima = cursor.fetchone()[0]
    print(
        "\n4. Validação Matemática do Faturamento: MAX(ABS((subtotal - discount_amount) - total)):"
    )
    print(f"   - Diferença máxima calculada no SQL: {diferenca_maxima:.15f}")
    if diferenca_maxima < 1e-5:
        print(
            "   - Conclusão: A equação de faturamento é 100% exata no banco relacional."
        )
    else:
        print(
            "   - Conclusão: A equação de faturamento não é 100% exata no banco relacional."
        )

    # 6. Cálculo de percentis via SQL (usando subqueries ordenadas com LIMIT/OFFSET)
    print("\n5. Cálculo de Percentis de 'total' (LIMIT/OFFSET ordenado):")
    percentis = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
    for p in percentis:
        offset = int(p * total_linhas)
        cursor.execute(
            f"SELECT total FROM orders ORDER BY total ASC LIMIT 1 OFFSET {offset};"
        )
        val = cursor.fetchone()[0]
        print(f"   - Percentil {p*100:5.1f}%: R$ {val:10.2f}")

    conn.close()


executar_diagnostico()

Questão 1 - Tarefa 3: Diagnóstico de confiabilidade dos dados na tabela 'orders'
------------------------------------------------------------------------

1. Verificação de Chave Primária ('id'):
   - IDs Duplicados: 0 (Garante unicidade dos registros)

2. Distribuição de Pedidos por Status:
   - paid: 34365 (70.14%)
   - confirmed: 7335 (14.97%)
   - cancelled: 4847 (9.89%)
   - draft: 2451 (5.00%)

3. Distribuição por Canal ('channel'):
   - ecommerce: 34342 (70.09%)
   - pos: 14656 (29.91%)

3.1. Análise de integridade dos dados por Canal por não ter salesperson_id agregado:
   - Canal: ecommerce  | Sem Id de Vendedor: 24131  | % do Total Geral de Pedidos: 49.25%
   - Canal: pos        | Sem Id de Vendedor: 0      | % do Total Geral de Pedidos: 0.0%

4. Validação Matemática do Faturamento: MAX(ABS((subtotal - discount_amount) - total)):
   - Diferença máxima calculada no SQL: 0.000000000014552
   - Conclusão: A equação de faturamento é 100% exata no banco relacional.

5. Cálculo de 

### Diagnóstico sobre a Confiabilidade da Tabela `orders` após Análise exploratória aprofundada

Com base na análise exploratória aprofundada realizada via SQL sobre os **48.998 registros** da tabela `orders`, apresento o seguinte diagnóstico técnico de confiabilidade dos dados:

#### 1. Distribuição da Coluna `total` e Avaliação de Outliers

* **Métricas Estatísticas:** A coluna `total` varia de **R$ 32,62** a **R$ 127.262,02**, com média de **R$ 28.704,99** e mediana de **R$ 25.918,02** ($P_{50}$). A média superior à mediana indica uma assimetria positiva moderada, impulsionada por vendas de alto valor nos percentis superiores ($P_{99} = \text{R\$ } 81.766,12$).
* **Critério Estatístico (IQR):** Pelo método do Intervalo Interquartil ($\text{IQR} = Q_3 - Q_1 = \text{R\$ } 40.941,93 - \text{R\$ } 13.170,56 = \text{R\$ } 27.771,37$), valores acima do limite de **R$ 82.598,99** ($Q_3 + 1,5 \times \text{IQR}$) configuram potenciais *outliers*.
* **Interpretação de Negócio:** Esses valores extremos **não devem ser removidos automaticamente**. Eles representam transações legítimas e plausíveis para a empresa por estar no mercado de varejo náutico, setor caracterizado pela existência de itens de baixo valor (peças e acessórios) e itens de alto valor agregado (motores e embarcações). Não foram identificados valores negativos, nulos ou inconsistências numéricas aparentes na coluna.

#### 2. Qualidade dos Dados e Integridade Estrutural

* **Completude e Nulos (`salesperson_id`):** Apenas a coluna `salesperson_id` possui valores nulos (**24.131 linhas**, ou **~49,25%** da tabela). A análise por canal revelou que **100% dos nulos estão concentrados no canal `ecommerce`** (correspondendo a **70,27%** dos pedidos online), enquanto no canal físico (`pos`) o preenchimento é de **100%** (0% nulos). Esse comportamento é condizente com fluxos digitais de autosserviço (sem vendedor direto), mas a regra deve ser formalmente validada com a área de negócios. As demais 12 colunas da tabela possuem **100% de preenchimento**.
* **Integridade Matemática:** A relação `total = subtotal - discount_amount` foi testada e confirmou integridade dos valores, com divergência máxima na ordem de apenas $1,45 \times 10^{-11}$ (sem nenhuma inconsistência superior a R$ 0,01).
* **Unicidade e Sequência de Identificadores:** As chaves `id` e `order_number` são únicas (zero duplicidades). A presença do registro máximo de `ID 50000` em uma base de 48.998 linhas indica **1.002 lacunas de IDs**. Em bancos relacionais (como PostgreSQL), sequências automáticas (*auto-increment/serial*) geram lacunas decorrentes de transações canceladas (*rollbacks*), quedas de conexão, pagamentos recusados ou exclusão de pedidos de teste/corrompidos. Isoladamente, isso não representa perda de dados, mas recomenda-se auditoria em logs caso seja necessário mapear o histórico.

#### 3. Distribuição por Status do Pedido

A base operacional divide-se em quatro principais estados de processamento:

* **Pagos (`paid`):** **70,14%** da base (**34.365 pedidos**), representando a receita efetivamente realizada.
* **Confirmados (`confirmed`):** **14,97%** da base (**7.335 pedidos**), (necessário confirmar o significado contábil e se afeta o faturamento total).
* **Cancelados (`cancelled`):** **9,89%** da base (**4.847 pedidos**), que não devem ser contabilizados como receita.
* **Rascunhos (`draft`):** **5,00%** da base (**2.451 pedidos**), referentes a carrinhos e orçamentos não finalizados.

> **Atenção:** Aproximadamente **14,89%** correspondem a soma de `cancelled` e `draft`, utilizá-los sem filtragem pode acabar inflando o faturamento total.

#### 4. Prontidão para Análises Futuras e Recomendações

A tabela `orders` **é estruturalmente confiável e consistente** para suportar as próximas etapas, desde que observadas as seguintes diretrizes:

1. **Filtragem Regrada por Status:** Não tratar a tabela bruta como relatório financeiro final. Para análises de faturamento e receita realizada, é mandatório aplicar filtros (prioritariamente `paid`, avaliando a inclusão de `confirmed` conforme a definição contábil do negócio).
2. **Preservação de Outliers:** Manter os valores elevados da coluna `total`, pois refletem o comportamento real do mercado náutico.
3. **Tratamento de Nulos:** Manter os nulos de `salesperson_id` documentados como transações de autosserviço do *e-commerce*.
4. **Cruzamento Relacional:** Para análises estratégicas mais ricas (como comportamento e fidelidade de clientes, cesta de compras, ticket médio por categoria), a tabela `orders` deverá ser cruzada com tabelas de suporte.

#### Observação Extra - Horizonte Temporal dos Dados

A coluna `created_at` cobre um intervalo entre **01/01/2020** e **31/12/2026**. Em um ambiente transacional estrito, datas no ano de 2026 sugeririam agendamentos ou falhas de sincronização do relógio do servidor. No entanto, dentro do escopo deste projeto/desafio, o intervalo de 2026 integra a base simulada da empresa e atua como conjunto de teste para a validação dos modelos preditivos das etapas seguintes.